# Week 4 Exercise — Unit Test Generator

Week 4 built a **code generator**: paste Python, pick a model, get high-performance C++ (or Rust), then compile and run.

For this exercise I picked one of the official follow-on ideas:

> **A code gen tool that writes unit test cases**

Same shape as the lab, different product:

1. Paste Python source
2. Choose an **open-weight** model (ranked on [Artificial Analysis](https://artificialanalysis.ai/models/open-source))
3. Generate `unittest` tests
4. Run the tests locally and read the report

I used **open-source / open-weight models only** via OpenRouter (OpenAI-compatible API, same pattern as `day4.ipynb` / `day5.ipynb`). No GPT/Claude/Gemini.

**Needs:** `OPENROUTER_API_KEY` in your `.env`

## Why this idea (and not the others)

| Idea | Why I skipped / picked it |
|---|---|
| Add more models to the C++ porter | Useful, but it is the same app as the lab |
| Agentic version | Better as a later stretch once generation + run works |
| Auto docstring / comments | Simpler, harder to *verify* quality |
| **Unit test generator** | Same generate-then-execute loop as Week 4, and the test report is a real quality signal |
| Simulated trading code | Fun, but needs a fake exchange API and is harder to grade |

Coding models that score well on Artificial Analysis's **Coding Index** are a good fit: they have to invent assertions, edge cases, and imports that actually run.

## Models from Artificial Analysis (open weights)

Source: [Open source models](https://artificialanalysis.ai/models/open-source) and [leaderboard](https://artificialanalysis.ai/leaderboards/models), with coding scores from OpenRouter's AA snapshots (Sep 2026).

| Model | OpenRouter id | AA notes | Why it is in the dropdown |
|---|---|---|---|
| **GLM 5.3 Flash** (Z.ai) | `z-ai/glm-5.3-flash` | Intelligence ~42, **Coding Index 71.5**, ~$0.10 blended | Default: strong coding, cheap, fast |
| **Qwen3.8 27B** (Alibaba) | `qwen/qwen3.8-27b` | Intelligence ~34 (xhigh), **Coding Index 68.1** | Dense 27B open-weight coder |
| **DeepSeek V4 Flash** | `deepseek/deepseek-v4-flash` | High coding / speed on the open-weight board | Cheap MoE, good for iteration |
| **MiMo-V2.6-Flash** (Xiaomi) | `xiaomi/mimo-v2.6-flash` | Open-source MoE, listed on AA open-source table | Extra open-weight option |

All four are **downloadable weights**, hosted here through OpenRouter so you do not need a local GPU. Swap ids if a provider is down.

In [ ]:
import os
import re
import sys
import tempfile
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from styles import CSS

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OPENROUTER_API_KEY is not set — add it to .env before generating tests")

openrouter = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
# Open-weight models only. Labels include the Artificial Analysis coding snapshot.
MODELS = {
    "GLM 5.3 Flash  |  AA Coding 71.5": "z-ai/glm-5.3-flash",
    "Qwen3.8 27B  |  AA Coding 68.1": "qwen/qwen3.8-27b",
    "DeepSeek V4 Flash  |  AA open-weight coder": "deepseek/deepseek-v4-flash",
    "MiMo-V2.6-Flash  |  AA open-source": "xiaomi/mimo-v2.6-flash",
}

MODEL_LABELS = list(MODELS.keys())
DEFAULT_MODEL = MODEL_LABELS[0]
MODELS

## Sample Python to test

A small shopping-cart helper with happy paths and edge cases (empty cart, unknown SKU, discounts, tax). The model should invent tests; it should not just echo this file.

In [ ]:
SAMPLE_PYTHON = '''from dataclasses import dataclass


@dataclass(frozen=True)
class LineItem:
    sku: str
    unit_price: float
    qty: int

    def subtotal(self) -> float:
        if self.qty < 0:
            raise ValueError("qty cannot be negative")
        if self.unit_price < 0:
            raise ValueError("unit_price cannot be negative")
        return round(self.unit_price * self.qty, 2)


class Cart:
    def __init__(self):
        self._items: dict[str, LineItem] = {}

    def add(self, sku: str, unit_price: float, qty: int = 1) -> None:
        if sku in self._items:
            current = self._items[sku]
            qty = current.qty + qty
            unit_price = current.unit_price
        self._items[sku] = LineItem(sku, unit_price, qty)

    def remove(self, sku: str) -> None:
        if sku not in self._items:
            raise KeyError(f"unknown sku: {sku}")
        del self._items[sku]

    def subtotal(self) -> float:
        return round(sum(item.subtotal() for item in self._items.values()), 2)

    def total(self, discount_pct: float = 0.0, tax_pct: float = 0.0) -> float:
        if not 0 <= discount_pct <= 100:
            raise ValueError("discount_pct must be between 0 and 100")
        if tax_pct < 0:
            raise ValueError("tax_pct cannot be negative")
        after_discount = self.subtotal() * (1 - discount_pct / 100)
        return round(after_discount * (1 + tax_pct / 100), 2)

    def skus(self) -> list[str]:
        return sorted(self._items)
'''

print(SAMPLE_PYTHON[:400], "...")

## Prompts

Same idea as the C++ porter: a tight system prompt, a user prompt that includes the source, and an instruction to **respond with code only**. Tests import from `app_under_test` so we can write two files and run `python -m unittest`.

In [ ]:
SYSTEM_PROMPT = """You are a senior Python engineer who writes thorough, runnable unit tests.

Rules:
- Respond with Python only. No markdown fences. No explanation before or after the code.
- Use the standard library `unittest` module (do not use pytest).
- Import production code with: from app_under_test import ...
- Do not copy the original source into the test file.
- Cover happy paths, edge cases, and expected exceptions.
- Use unittest.TestCase methods (assertEqual, assertRaises, assertAlmostEqual, ...).
- End the file so it is runnable with: python -m unittest
"""


def user_prompt_for(python_source: str) -> str:
    return f"""Write unittest tests for the Python module below.

The module will be saved as app_under_test.py. Your tests will be saved as test_app.py
and executed with: python -m unittest test_app -v

Respond only with the test file contents.

```python
{python_source}
```
"""


def messages_for(python_source: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_for(python_source)},
    ]


FENCE_RE = re.compile(r"```(?:python)?\s*([\s\S]*?)```", re.IGNORECASE)


def extract_python(text: str) -> str:
    """Strip markdown fences if the model ignores the 'code only' rule."""
    if not text:
        return ""
    blocks = FENCE_RE.findall(text)
    if blocks:
        return blocks[0].strip()
    return text.replace("```python", "").replace("```", "").strip()

In [ ]:
def generate_tests(model_label: str, python_source: str):
    """Stream generated tests into the Gradio code box."""
    if not openrouter_api_key:
        yield "", "Set OPENROUTER_API_KEY in .env first."
        return
    if not python_source or not python_source.strip():
        yield "", "Paste some Python source first."
        return

    model_id = MODELS[model_label]
    collected = ""
    try:
        stream = openrouter.chat.completions.create(
            model=model_id,
            messages=messages_for(python_source),
            stream=True,
            extra_body={"reasoning": {"effort": "low"}},
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            if delta:
                collected += delta
                yield collected, f"Generating with {model_id}..."
        code = extract_python(collected)
        yield code, f"Done — {model_id}"
    except Exception as exc:
        yield collected, f"Generation failed: {exc}"


def run_tests(python_source: str, test_source: str) -> str:
    """Write source + tests to a temp folder and run unittest (timeout 20s)."""
    if not python_source.strip():
        return "No Python source to test."
    if not test_source.strip():
        return "No tests yet — generate them first."

    test_source = extract_python(test_source)
    with tempfile.TemporaryDirectory(prefix="wk4_tests_") as tmp:
        tmp_path = Path(tmp)
        (tmp_path / "app_under_test.py").write_text(python_source, encoding="utf-8")
        (tmp_path / "test_app.py").write_text(test_source, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, "-m", "unittest", "test_app", "-v"],
                cwd=tmp,
                capture_output=True,
                text=True,
                timeout=20,
            )
        except subprocess.TimeoutExpired:
            return "Timed out after 20s — tests may have an infinite loop."

        parts = [
            f"exit code: {result.returncode}",
            "--- stdout ---",
            result.stdout.strip() or "(empty)",
            "--- stderr ---",
            result.stderr.strip() or "(empty)",
        ]
        return "\n".join(parts)

## Gradio UI

Mirrors the Week 4 porting UI: source on the left, generated code on the right, run buttons underneath. Pick a model from the Artificial Analysis open-weight set, generate, then run.

In [ ]:
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Week 4 — Unit Test Generator") as ui:
    gr.Markdown(
        "### Week 4 exercise — unit test generator\n"
        "Open-weight models only, chosen from [Artificial Analysis](https://artificialanalysis.ai/models/open-source). "
        "Paste Python, generate `unittest` tests, then run them."
    )
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_box = gr.Code(
                label="Python source",
                value=SAMPLE_PYTHON,
                language="python",
                lines=26,
            )
        with gr.Column(scale=6):
            tests_box = gr.Code(
                label="Generated unittest file",
                value="",
                language="python",
                lines=26,
            )

    with gr.Row(elem_classes=["controls"]):
        model_dd = gr.Dropdown(MODEL_LABELS, value=DEFAULT_MODEL, label="Open-weight model")
        generate_btn = gr.Button("Generate tests", elem_classes=["convert-btn"])
        run_btn = gr.Button("Run tests", elem_classes=["run-btn", "py"])

    with gr.Row():
        status = gr.Textbox(label="Status", lines=1)
    with gr.Row():
        report = gr.TextArea(label="unittest report", lines=14, elem_classes=["py-out"])

    generate_btn.click(
        fn=generate_tests,
        inputs=[model_dd, python_box],
        outputs=[tests_box, status],
    )
    run_btn.click(
        fn=run_tests,
        inputs=[python_box, tests_box],
        outputs=[report],
    )

ui.launch(inbrowser=True)

## How to try it

1. Confirm `OPENROUTER_API_KEY` is in `.env`.
2. Run all cells. Gradio opens in the browser.
3. Keep the sample cart, or paste your own module.
4. Start with **GLM 5.3 Flash**, generate, then **Run tests**.
5. Compare with Qwen3.8 27B / DeepSeek V4 Flash: look at coverage of edge cases (`qty < 0`, unknown SKU, discount bounds) and whether `unittest` actually passes.

If a model wraps the answer in ` ```python `, `extract_python` strips that before saving.

### Stretch ideas (same Week 4 prompt)

- Ask a second model to **review** failing tests and patch them (light agentic loop).
- Add a docstring/comment pass as a second action in the dropdown.
- Swap in a local Ollama coder (`qwen2.5-coder`) the way `day4.ipynb` does.